In [ ]:
aa# upload multilingual Dataset Toxic

from google.colab import files
files.upload()


Saving multilingual_toxicity.csv to multilingual_toxicity.csv


{'multilingual_toxicity.csv': b'text,toxic\n" Untaten mit ""Fl\xc3\xbcchtlingen"" oder Moslems in Verbindung zu bringen wird immer als rassistische Hetze gemeldet.  Fakten benennen = Rechtsextremismus. Da helfen nur blumige Umschreibungen. In der DDR hatte man auch Geheimsprachen entwickelt..",0\n"Der Regenwald wird im Akkord abgeholzt ,das interessiert niemanden !",0\n Es geht darum was der Makler sich rein zieht.,0\n" Huhhhhh, Fritz, Du jagst mir richtig Angst ein \xf0\x9f\x98\x9c",0\n"   Hab versucht zu entziffern was du da von dir gibst, bin dabei behindert geworden du Bastard",1\n"Wunderbar, wie die #Bundesdrogenkriegsbeauftragte sich als verlogenes Mistst\xc3\xbcck outet.\n\xf0\x9f\xa5\xa6\n#NieMehrCDUCSU \n#Cannabis \n#LegalizeIt \n#Regulierung\n#Jugendschutz\n#Verbraucherschutz https://t.co/1gn6wczu9F",1\nAuch durch eure Zensur in Deutschland werdet ihr nichts \xc3\xa4ndern. Wie sehr in Deutschland zensiert wird kann ich hier im Ausland sehr gut erkennen. |LBR| #illner,0\n"    

In [ ]:
!pip install transformers datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 17.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
!pip install transformers datasets sklearn
import os
from transformers import BertConfig, BertForSequenceClassification
import pandas as pd
import torch
from datasets import Dataset
from transformers import BertTokenizerFast, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.model_selection import train_test_split

# Ensure GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Step 1: Load dataset
df = pd.read_csv("multilingual_toxicity.csv")
df = df.dropna(subset=["text", "toxic"])  # Remove missing values
df["toxic"] = df["toxic"].astype(int)  # Ensure labels are integers

# Step 2: Split dataset
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["toxic"])
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Step 3: Load tokenizer and model
model_name = "bert-base-multilingual-cased"
config = BertConfig.from_pretrained(
    model_name,
    hidden_dropout_prob=0.2,  # Increase dropout for fully connected layers
    attention_probs_dropout_prob=0.2  # Increase dropout for attention probabilities
)
model = BertForSequenceClassification.from_pretrained("bert-base-multilingual-cased", config=config).to(device)

tokenizer = BertTokenizerFast.from_pretrained(model_name)

# Step 4: Tokenize datasets
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_tokenized_dataset = train_dataset.map(tokenize_function, batched=True)
test_tokenized_dataset = test_dataset.map(tokenize_function, batched=True)

# Remove unnecessary columns
train_tokenized_dataset = train_tokenized_dataset.remove_columns(["text", "__index_level_0__"])
test_tokenized_dataset = test_tokenized_dataset.remove_columns(["text", "__index_level_0__"])

# Rename labels for Trainer compatibility
train_tokenized_dataset = train_tokenized_dataset.rename_column("toxic", "labels")
test_tokenized_dataset = test_tokenized_dataset.rename_column("toxic", "labels")

# Set format for PyTorch
train_tokenized_dataset.set_format("torch")
test_tokenized_dataset.set_format("torch")

# Step 5: Define metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
    }

# Step 6: Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=7,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",  # Evaluate after every epoch
    save_strategy="epoch",  # Save model checkpoint after each epoch
    learning_rate=5e-6,  # Set a smaller learning rate for fine-tuning
    weight_decay=0.01,  # Add weight decay for regularization
    warmup_steps=500,  # Gradually increase learning rate in the first 500 steps
    logging_steps=50,  # Log metrics less frequently
    load_best_model_at_end=True,  # Load the best checkpoint
    save_total_limit=2,  # Keep only the last 2 checkpoints
    metric_for_best_model="eval_loss",  # Use validation loss to determine the best model
    greater_is_better=False,  # Smaller validation loss is better
    report_to=[]  # Disable reporting to external services like WandB
)

# Step 7: Add early stopping callback
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized_dataset,
    eval_dataset=test_tokenized_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],  # Stop if no improvement in 3 epochs
)

# Step 8: Train the model
trainer.train()

# Step 9: Evaluate the model
eval_results = trainer.evaluate()
print("Final Evaluation Results:")
print(f"Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"F1-Score: {eval_results['eval_f1']:.4f}")
print(f"Precision: {eval_results['eval_precision']:.4f}")
print(f"Recall: {eval_results['eval_recall']:.4f}")


  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Using device: cuda


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.692500,0.670692,0.656000,0.652091,0.663341,0.656000
2,0.583700,0.536536,0.789000,0.788797,0.790115,0.789000
3,0.408600,0.357894,0.850000,0.849528,0.854446,0.850000
4,0.346200,0.302500,0.875000,0.874985,0.875182,0.875000
5,0.235200,0.314372,0.876000,0.875389,0.883517,0.876000
6,0.272600,0.301911,0.891000,0.890798,0.893913,0.891000
7,0.188100,0.280290,0.887000,0.887000,0.887002,0.887000
8,0.150400,0.317412,0.890000,0.889548,0.896496,0.890000
9,0.162700,0.317999,0.888000,0.887719,0.891919,0.888000


Final Evaluation Results:
Accuracy: 0.8870
F1-Score: 0.8870
Precision: 0.8870
Recall: 0.8870
